In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm
from statsmodels.tsa.stattools import grangercausalitytests, adfuller

In [2]:
DATA_DIR = Path(".")
OUT_DIR = Path("./outputs")
OUT_DIR.mkdir(exist_ok=True)

In [3]:
START = pd.Timestamp("2024-01-01")
END   = pd.Timestamp("2025-03-31")

In [4]:
# Regime change anchors for plotting / event-study comments
DENCUN  = pd.Timestamp("2024-03-13")
ETF_19B = pd.Timestamp("2024-05-23")
ETF_LIVE = pd.Timestamp("2024-07-23")
AUG_CRASH = pd.Timestamp("2024-08-05")

### Loading each file

In [ ]:

# Loading data - ETH price (Yahoo) — has two metadata rows at the top

eth = pd.read_csv(DATA_DIR / "ether_data.csv", skiprows=3, header=None,
                  names=["date","eth_close","eth_high","eth_low","eth_open","eth_volume"])
eth["date"] = pd.to_datetime(eth["date"])
eth = eth[(eth["date"] >= START) & (eth["date"] <= END)].copy()
print(f"ETH:        {len(eth)} rows, {eth['date'].min().date()} -> {eth['date'].max().date()}")



In [ ]:
# Loading data - stETH price (CoinGecko, full history)
steth = pd.read_csv(DATA_DIR / "stETH-usd-max.csv")
steth["date"] = pd.to_datetime(steth["snapped_at"].str.replace(" UTC","")).dt.normalize()
steth = steth[(steth["date"] >= START) & (steth["date"] <= END)].copy()
steth = steth.rename(columns={"price":"steth_close","total_volume":"steth_volume"})[["date","steth_close","steth_volume"]]
print(f"stETH:      {len(steth)} rows, {steth['date'].min().date()} -> {steth['date'].max().date()}")

In [ ]:
# Loading data - Risk-free
rf = pd.read_csv(DATA_DIR / "DTB3.csv")
rf["date"] = pd.to_datetime(rf["observation_date"])
rf = rf[(rf["date"] >= START) & (rf["date"] <= END)].rename(columns={"DTB3":"rf_3m_pct"})[["date","rf_3m_pct"]]
print(f"Risk-free:  {len(rf)} rows (business days only — will be ffilled in merge)")

In [ ]:
# Loading data -  Gas (Etherscan, Wei -> Gwei)
gas = pd.read_csv(DATA_DIR / "export-AvgGasPrice.csv")
gas.columns = [c.strip() for c in gas.columns]
gas["date"] = pd.to_datetime(gas["Date(UTC)"], format="%m/%d/%Y")
gas["gas_avg_gwei"] = pd.to_numeric(gas["Value (Wei)"]) / 1e9
gas = gas[(gas["date"] >= START) & (gas["date"] <= END)][["date","gas_avg_gwei"]].copy()
print(f"Gas:        {len(gas)} rows")

In [ ]:
# Loading data - Lido APR (DefiLlama)
lido = pd.read_csv(DATA_DIR / "lido_apr_filtered.csv")
lido["date"] = pd.to_datetime(lido["timestamp"]).dt.tz_convert(None).dt.normalize()
lido = lido[(lido["date"] >= START) & (lido["date"] <= END)].copy()
lido = lido.rename(columns={"apy":"lido_apr_pct","tvlUsd":"lido_tvl_usd"})[["date","lido_apr_pct","lido_tvl_usd"]]
print(f"Lido:       {len(lido)} rows")

In [ ]:
# Loading data - Uniswap pool
uni = pd.read_csv(DATA_DIR / "uniswap_pool_v2.csv")
uni["date"] = pd.to_datetime(uni["date"]).dt.normalize()
uni = uni.sort_values("date").reset_index(drop=True)
uni = uni[(uni["date"] >= START) & (uni["date"] <= END)].copy()
print(f"Uniswap:    {len(uni)} rows, {uni['date'].min().date()} -> {uni['date'].max().date()}")

### Merge into daily panel

In [ ]:
panel = pd.DataFrame({"date": pd.date_range(START, END, freq="D")})
for df_ in [eth, steth, rf, gas, lido, uni]:
    panel = panel.merge(df_, on="date", how="left")

In [ ]:
# Forward-fill risk-free across weekends/holidays
panel["rf_3m_pct"] = panel["rf_3m_pct"].ffill()

In [ ]:
# Sanity report on missing values
print("\nMissing values per column (before any imputation):")
print(panel.isna().sum().to_string())

### Calculate metrics

In [ ]:
p = panel.copy()

In [ ]:
# ETH returns and vol
p["eth_simple_return"]  = p["eth_close"].pct_change()
p["eth_log_return"]     = np.log(p["eth_close"]).diff()
p["eth_realised_vol_30d"] = p["eth_log_return"].rolling(30).std() * np.sqrt(365)
p["eth_sigma_daily"]    = p["eth_realised_vol_30d"] / np.sqrt(365)

In [ ]:
# --- stETH ---
# IMPORTANT: ETH price (Yahoo Finance) and stETH price (CoinGecko) are snapped at
# different times of day. Yahoo's daily "Close" reflects ~21:00 UTC (US market close),
# while CoinGecko snaps at 00:00 UTC.

# Empirically this means stETH(t) corresponds to
# ETH(t-1). Without this alignment the peg deviation series shows spurious ±15% spikes
# on high-volatility days. We therefore align stETH(t) to ETH(t-1) for the peg ratio

p["eth_close_prev"]      = p["eth_close"].shift(1)
p["steth_simple_return"] = p["steth_close"].pct_change()
p["steth_eth_ratio"]     = p["steth_close"] / p["eth_close_prev"]
p["steth_peg_bps"]       = 10000 * (p["steth_eth_ratio"] - 1)
p["steth_daily_yield"]   = p["lido_apr_pct"] / 100 / 365

In [ ]:
# LP economics
# Without daily TVL we use a constant proxy: $200M (median LP TVL for this pool over the sample,
# documented as a methodology limitation). This is the price paid for skipping the TVL series.

TVL_PROXY_USD = 200_000_000  # discuss in methodology
p["uni_fee_apr_daily"]    = p["fees_usd"] / TVL_PROXY_USD
p["uni_fee_apr_annualised"] = p["uni_fee_apr_daily"] * 365

In [ ]:
# LVR (constant-product proxy): sigma^2 / 8

p["lvr_daily"] = p["eth_sigma_daily"]**2 / 8

In [ ]:
# LP edge

p["lp_edge_daily"] = p["uni_fee_apr_daily"] - p["lvr_daily"]

### Three Startegies — Cumulative returns from $1

In [ ]:
# HODL ETH

p["strat_HODL"] = (1 + p["eth_simple_return"].fillna(0)).cumprod()

In [ ]:
# stETH (its market price already incorporates staking yield via daily rebase)
p["strat_stETH"] = (1 + p["steth_simple_return"].fillna(0)).cumprod()

In [ ]:
#  Uniswap v3 LP

p["lp_daily_return"] = (
    0.5 * p["eth_simple_return"].fillna(0)
    + p["uni_fee_apr_daily"].fillna(0)
    - p["lvr_daily"].fillna(0)
)
p["strat_LP"] = (1 + p["lp_daily_return"]).cumprod()

In [ ]:
print(f"{'Strategy':<15}  {'Final $':<10}  {'Total return':<15}")
for s in ["strat_HODL","strat_stETH","strat_LP"]:
    final = p[s].dropna().iloc[-1]
    print(f"{s:<15}  {final:>9.4f}  {(final-1)*100:>10.2f}%")

Performance summary (annualised return, vol, Sharpe, max DD)

In [ ]:

print("\n" + "="*70)
print("Step 5: Performance summary")
print("="*70)

In [ ]:
def perf(returns, rf_daily, label):
    r = returns.dropna()
    rf_aligned = rf_daily.reindex(r.index).fillna(0)
    excess = r - rf_aligned
    n = len(r)
    ann_ret = (1+r).prod()**(365/n) - 1 if n > 0 else np.nan
    ann_vol = r.std() * np.sqrt(365)
    sharpe = (excess.mean() / r.std() * np.sqrt(365)) if r.std() > 0 else np.nan
    cum = (1+r).cumprod()
    dd = (cum / cum.cummax() - 1).min()
    return {"strategy": label, "ann_return": ann_ret, "ann_vol": ann_vol,
            "sharpe": sharpe, "max_drawdown": dd, "n_obs": n}

In [ ]:
p_idx = p.set_index("date")
rf_daily = p_idx["rf_3m_pct"]/100/365

In [ ]:
ret_HODL  = p_idx["strat_HODL"].pct_change()
ret_stETH = p_idx["strat_stETH"].pct_change()
ret_LP    = p_idx["strat_LP"].pct_change()

In [ ]:
summary = pd.DataFrame([
    perf(ret_HODL,  rf_daily, "HODL ETH"),
    perf(ret_stETH, rf_daily, "stETH (Lido)"),
    perf(ret_LP,    rf_daily, "Uniswap v3 LP (delta-hedged proxy)"),
])
summary.to_csv(OUT_DIR / "strategy_summary.csv", index=False)
print(summary.to_string(index=False))

### Regressions

In [ ]:
reg_log = []
def log(s):
    print(s)
    reg_log.append(str(s))

In [ ]:
# Model 1: LP edge ~ vol + gas + |return|
reg = p.dropna(subset=["lp_edge_daily","eth_realised_vol_30d","gas_avg_gwei","eth_simple_return"]).copy()
X = pd.DataFrame({
    "eth_vol_30d": reg["eth_realised_vol_30d"],
    "gas_gwei":    reg["gas_avg_gwei"],
    "abs_eth_ret": reg["eth_simple_return"].abs(),
})
X = sm.add_constant(X)
y = reg["lp_edge_daily"]
m1 = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
log("\n--- Model 1: LP edge ~ vol + gas + |ETH return| (HAC, lag=5) ---")
log(m1.summary().as_text())

In [ ]:
# Model 2: AR(1) on stETH peg deviation
peg = p[["date","steth_peg_bps"]].dropna().copy()
peg["peg_lag1"] = peg["steth_peg_bps"].shift(1)
peg = peg.dropna()
Xp = sm.add_constant(peg[["peg_lag1"]])
m2 = sm.OLS(peg["steth_peg_bps"], Xp).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
log("\n--- Model 2: AR(1) stETH peg deviation (HAC, lag=5) ---")
log(m2.summary().as_text())
phi = m2.params["peg_lag1"]
if 0 < phi < 1:
    log(f"\nHalf-life of peg deviation: {-np.log(2)/np.log(phi):.2f} days")
else:
    log(f"\nAR coefficient = {phi:.3f} — outside (0,1), interpret with care")

In [ ]:
# ADF test for peg stationarity
adf = adfuller(peg["steth_peg_bps"].values)
log(f"\nADF test on peg: stat={adf[0]:.3f}, p-value={adf[1]:.4f}, lags={adf[2]}")

In [ ]:
# Model 3: Granger causality
log("\n--- Model 3a: Granger ETH vol -> |stETH peg deviation| ---")
gc_data = p[["eth_realised_vol_30d","steth_peg_bps"]].dropna().copy()
gc_data["abs_peg"] = gc_data["steth_peg_bps"].abs()
gc1 = grangercausalitytests(gc_data[["abs_peg","eth_realised_vol_30d"]], maxlag=5, verbose=False)
for lag, res in gc1.items():
    pval = res[0]["ssr_ftest"][1]
    log(f"  lag {lag}: F p-value = {pval:.4f}")

In [ ]:
log("\n--- Model 3b: Granger |stETH peg deviation| -> ETH vol ---")
gc2 = grangercausalitytests(gc_data[["eth_realised_vol_30d","abs_peg"]], maxlag=5, verbose=False)
for lag, res in gc2.items():
    pval = res[0]["ssr_ftest"][1]
    log(f"  lag {lag}: F p-value = {pval:.4f}")

In [ ]:
# Save full regression log
with open(OUT_DIR / "regression_results.txt", "w") as f:
    f.write("\n".join(reg_log))

Key events and extremes (for the discussion section)

In [ ]:

print("\n" + "="*70)
print("Step 7: Notable extremes")
print("="*70)

In [ ]:
# Largest peg deviations
peg_sorted = p[["date","steth_peg_bps"]].dropna().sort_values("steth_peg_bps")
print("Most negative stETH peg deviations:")
print(peg_sorted.head(5).to_string(index=False))
print("Most positive stETH peg deviations:")
print(peg_sorted.tail(5).to_string(index=False))

In [ ]:
# Largest LP edge swings
edge_sorted = p[["date","lp_edge_daily"]].dropna().sort_values("lp_edge_daily")
print("\nMost negative LP edge days:")
print(edge_sorted.head(5).to_string(index=False))
print("Most positive LP edge days:")
print(edge_sorted.tail(5).to_string(index=False))

In [ ]:
# Largest single day ETH moves
mv = p[["date","eth_simple_return"]].dropna().copy()
mv["abs_ret"] = mv["eth_simple_return"].abs()
print("\nLargest absolute single-day ETH moves:")
print(mv.nlargest(5, "abs_ret")[["date","eth_simple_return"]].to_string(index=False))

In [ ]:
# Vol regime stats
print(f"\nETH 30d realised vol: mean={p['eth_realised_vol_30d'].mean():.3f}, "
      f"max={p['eth_realised_vol_30d'].max():.3f} on {p.loc[p['eth_realised_vol_30d'].idxmax(),'date'].date() if not p['eth_realised_vol_30d'].isna().all() else 'NaN'}, "
      f"min={p['eth_realised_vol_30d'].min():.3f}")

### Charts

In [ ]:

print("\n" + "="*70)
print("Step 8: Generating charts")
print("="*70)

In [ ]:
def add_event_lines(ax):
    for d, lbl, col in [(DENCUN,"Dencun","grey"),
                         (ETF_LIVE,"ETH ETF","green"),
                         (AUG_CRASH,"Aug crash","red")]:
        ax.axvline(d, color=col, linestyle="--", alpha=0.5, linewidth=1)
        ax.text(d, ax.get_ylim()[1], lbl, rotation=90, va="top", ha="right", fontsize=8, color=col)

In [ ]:
# Figure 1 - cumulative returns of three strategies
fig, ax = plt.subplots(figsize=(11,5.5))
p.set_index("date")[["strat_HODL","strat_stETH","strat_LP"]].plot(ax=ax)
ax.set_title("Cumulative return from $1: ETH HODL vs stETH vs Uniswap v3 LP\n(Jan 2024 – Mar 2025)")
ax.set_ylabel("NAV index, $1 = 1 Jan 2024")
ax.set_xlabel("")
ax.legend(["HODL ETH","stETH (Lido)","Uniswap v3 LP"])
ax.grid(alpha=0.3)
add_event_lines(ax)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_cumulative_returns.png", dpi=150)
plt.close()
print(" - fig_cumulative_returns.png")

In [ ]:
# Figure 2 - ETH price + 30d realised vol
fig, ax1 = plt.subplots(figsize=(11,5))
ax1.plot(p["date"], p["eth_close"], color="steelblue", label="ETH price (USD, left)")
ax1.set_ylabel("ETH price (USD)", color="steelblue")
ax2 = ax1.twinx()
ax2.plot(p["date"], p["eth_realised_vol_30d"], color="firebrick", label="30d realised vol (right)")
ax2.set_ylabel("30d realised vol (annualised)", color="firebrick")
ax1.set_title("ETH price and 30-day realised volatility (Jan 2024 – Mar 2025)")
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_eth_price_vol.png", dpi=150)
plt.close()
print(" - fig_eth_price_vol.png")

In [ ]:
# Figure 3 - LP economics
fig, ax = plt.subplots(figsize=(11,5))
roll = 7
ax.plot(p["date"], p["uni_fee_apr_daily"].rolling(roll).mean()*365, label="Annualised fee APR (7d MA)", color="seagreen")
ax.plot(p["date"], p["lvr_daily"].rolling(roll).mean()*365, label="Annualised LVR proxy (7d MA)", color="firebrick")
ax.plot(p["date"], p["lp_edge_daily"].rolling(roll).mean()*365, label="LP edge (fee - LVR, 7d MA)", color="navy", linewidth=2)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_title("Uniswap v3 ETH/USDC 0.05% pool: fee yield, LVR, and net LP edge (annualised)")
ax.set_ylabel("Annualised yield")
ax.legend()
ax.grid(alpha=0.3)
add_event_lines(ax)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_lp_economics.png", dpi=150)
plt.close()
print(" - fig_lp_economics.png")

In [ ]:
# Figure 4 - stETH peg deviation
fig, ax = plt.subplots(figsize=(11,5))
ax.plot(p["date"], p["steth_peg_bps"], color="purple", linewidth=1)
ax.axhline(0, color="black", linewidth=0.5)
ax.fill_between(p["date"], p["steth_peg_bps"], 0,
                where=(p["steth_peg_bps"]<0), color="red", alpha=0.2, label="discount")
ax.fill_between(p["date"], p["steth_peg_bps"], 0,
                where=(p["steth_peg_bps"]>=0), color="green", alpha=0.2, label="premium")
ax.set_title("stETH/ETH peg deviation (basis points)")
ax.set_ylabel("Deviation from 1.0 (bps)")
ax.legend()
ax.grid(alpha=0.3)
add_event_lines(ax)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_steth_peg.png", dpi=150)
plt.close()
print(" - fig_steth_peg.png")

### Save panel

In [ ]:
p.to_csv(OUT_DIR / "master_panel.csv", index=False)
print(f"\nMaster panel saved: {OUT_DIR/'master_panel.csv'} ({len(p)} rows × {p.shape[1]} cols)")